**🕰️ What is Time Travel in Delta Lake?**
Time Travel allows you to query previous versions of a Delta table. This is useful for:
- •	Auditing changes
- •	Debugging data issues
- •	Recovering accidentally deleted data
- •	Comparing historical data

Delta Lake automatically tracks version history every time you write to a table (like INSERT, UPDATE, MERGE, etc.)

**🔍 How It Works**
Delta Lake stores metadata and data changes in a transaction log (_delta_log). Each change creates a new version of the table.
You can access older versions using:
- •	Version number
- •	Timestamp


Note: In Unity Catalog, use a managed table or a valid cloud storage path (abfss://...). The /FileStore/... location shown in older examples may not work.

### create table

In [0]:
%sql
CREATE OR REPLACE TABLE tri.tri.scd2Demo (
    pk1 INT,
    pk2 STRING,
    dim1 INT,
    dim2 INT,
    dim3 INT,
    dim4 INT,
    active_status STRING,
    start_date TIMESTAMP,
    end_date TIMESTAMP
)
USING DELTA;

In [0]:
%sql
INSERT INTO tri.tri.scd2Demo
VALUES
(101, 'EMP001', 10, 20, 30, 40, 'Y',
 TIMESTAMP('2024-01-01 00:00:00'),
 TIMESTAMP('9999-12-31 23:59:59'))




In [0]:
%sql
select * from tri.tri.scd2Demo

In [0]:
%sql
INSERT INTO tri.tri.scd2Demo
VALUES(102, 'EMP002', 15, 25, 35, 45, 'Y',
 TIMESTAMP('2024-01-15 00:00:00'),
 TIMESTAMP('9999-12-31 23:59:59'))

In [0]:
%sql
select *from tri.tri.scd2Demo

In [0]:
%sql
INSERT INTO tri.tri.scd2Demo values
(103, 'EMP003', 12, 22, 32, 42, 'Y',
 TIMESTAMP('2024-02-01 00:00:00'),
 TIMESTAMP('9999-12-31 23:59:59'));

In [0]:
%sql
select *from tri.tri.scd2Demo

In [0]:
%sql
describe history tri.tri.scd2Demo

### Method 1: PySpark Time Travel Using Timestamp + Table Name

In [0]:
df = spark.read \
    .format("delta") \
    .option("timestampAsOf", "2026-10-25T10:12:36.000+00:00") \
    .table("tri.tri.scd2Demo")

display(df)

### Method 2: PySpark Time Travel Using Timestamp + Path


In [0]:
df = spark.read \
    .format("delta") \
    .option("timestampAsOf", "2026-10-25T10:12:36.000+00:00") \
    .load("/Volumes/tri/tri/tri_files/scd2Demo")

display(df)

### Method 3: PySpark Time Travel Using Version + Table Name

In [0]:
df = spark.read \
    .format("delta") \
    .option("versionAsOf", 2) \
    .table("tri.tri.scd2Demo")

display(df)

### Method 4: PySpark Time Travel Using Version + Path


In [0]:
df = spark.read \
    .format("delta") \
    .option("versionAsOf", 3) \
    .load("/Volumes/tri/tri/tri_files/scd2Demo")

display(df)

path will not work

### Method 5: SQL Time Travel Using Version + Table Name

In [0]:
%sql
SELECT *
FROM tri.tri.scd2Demo
VERSION AS OF 2;

### Method 6: SQL Time Travel Using Version + Path

In [0]:
%sql
SELECT *
FROM delta.`/Volumes/tri/tri/tri_files/scd2Demo`
VERSION AS OF 2;


### Method 7: SQL Time Travel Using Timestamp + Table Name

In [0]:
%sql
SELECT *
FROM tri.tri.scd2Demo
TIMESTAMP AS OF '2026-10-25T10:12:36.000+00:00';

### Method 8: SQL Time Travel Using Timestamp + Path

In [0]:
%sql
SELECT *
FROM delta.`/Volumes/tri/tri/tri_files/scd2Demo`
TIMESTAMP AS OF '2024-01-15T10:00:00.000+0000';

In [0]:
%sql
DESCRIBE DETAIL tri.tri.scd2Demo;

**NOTE**

INSERT INTO scd2Demo
VALUES
(101, 'EMP001', 10, 20, 30, 40, 'Y',
 TIMESTAMP('2024-01-01 00:00:00'),
 TIMESTAMP('9999-12-31 23:59:59')),

(102, 'EMP002', 15, 25, 35, 45, 'Y',
 TIMESTAMP('2024-01-15 00:00:00'),
 TIMESTAMP('9999-12-31 23:59:59')),

(103, 'EMP003', 12, 22, 32, 42, 'Y',
 TIMESTAMP('2024-02-01 00:00:00'),
 TIMESTAMP('9999-12-31 23:59:59'));

 In Delta Lake, version numbers are assigned per transaction/commit, not per row.
Since your statement is a single INSERT command:
all 3 records are written in one transaction, so they belong to the same Delta version.